# 05 — Train 1D CNN (optional)

**This notebook is optional.** RF is the primary tabular baseline; the CNN exists to give the writeup a "raw signals vs. engineered features" comparison if/when there's time and hardware.

**To run this:**
1. Re-run `02_build_features.ipynb` with `WRITE_SIGNAL_CACHE = True` to produce `signals.h5`.
2. Run `03_train_rf.ipynb` first to produce `splits.json` (the CNN reuses the same subject-level test split).
3. On a Macbook, this falls back to MPS (Apple GPU) or CPU — expect ~minutes to tens-of-minutes per epoch depending on dataset size. On the 4070 it's ~2–5 min per epoch with the full dataset.

If `signals.h5` doesn't exist, the indexing cell will fail clearly — that's your cue to flip the flag in notebook 02.

In [ ]:
EPOCHS = 15
BATCH_SIZE = 256
LR = 1e-3
NUM_WORKERS = 4

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import json
import numpy as np
import torch
from torch.utils.data import DataLoader

from bme_ml.paths import setup_paths
from bme_ml.splits import Splits
from bme_ml.cnn import (
    SegmentDataset, build_indices_for_subjects, CNN1D, TrainConfig, train, predict_proba,
)
from bme_ml.evaluation import evaluate_binary

paths = setup_paths()

# Pick the best available device. CUDA wins (RTX 4070); else MPS (Apple
# Silicon); else CPU. NUM_WORKERS=0 on MPS — multi-worker dataloading is
# flaky with MPS as of late 2024/early 2025.
if torch.cuda.is_available():
    device = torch.device('cuda')
    print('device: cuda /', torch.cuda.get_device_name(0))
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    device = torch.device('mps')
    NUM_WORKERS = 0
    print('device: mps (Apple Silicon)')
else:
    device = torch.device('cpu')
    print('device: cpu (training will be slow — consider lowering EPOCHS)')

In [ ]:
splits = Splits.from_json(paths.splits_json)
# Use fold 0's val subjects as validation; the remaining train + other folds' val
# subjects fold into the CNN train set. This keeps the RF and CNN apples-to-apples
# on the test set without us re-doing the split.
fold0 = splits.cv_folds[0]
train_subj = list(fold0['train'])
val_subj = list(fold0['val'])
test_subj = splits.test_subjects
print(f'subjects  train={len(train_subj)}  val={len(val_subj)}  test={len(test_subj)}')

train_idx, train_y = build_indices_for_subjects(paths.signals_h5, train_subj)
val_idx,   val_y   = build_indices_for_subjects(paths.signals_h5, val_subj)
test_idx,  test_y  = build_indices_for_subjects(paths.signals_h5, test_subj)
print(f'segments  train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}')

ds_train = SegmentDataset(paths.signals_h5, train_idx, train_y)
ds_val   = SegmentDataset(paths.signals_h5, val_idx, val_y)
ds_test  = SegmentDataset(paths.signals_h5, test_idx, test_y)

common = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))
dl_train = DataLoader(ds_train, shuffle=True,  **common)
dl_val   = DataLoader(ds_val,   shuffle=False, **common)
dl_test  = DataLoader(ds_test,  shuffle=False, **common)

In [ ]:
# Class weights: counter the MIMIC-derived hypertension over-representation.
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_y)
class_weights_t = torch.tensor(class_weights, dtype=torch.float32)
print('class weights:', class_weights)

model = CNN1D(in_channels=2, n_classes=2)
n_params = sum(p.numel() for p in model.parameters())
print(f'CNN parameters: {n_params/1e3:.1f} k')

cfg = TrainConfig(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, num_workers=NUM_WORKERS)
model = train(model, dl_train, dl_val, cfg, device, class_weights=class_weights_t)

In [ ]:
y_true, y_pred, y_proba = predict_proba(model, dl_test, device)
metrics = evaluate_binary(y_true, y_pred, y_proba)
print(metrics)

torch.save(model.state_dict(), paths.models / 'cnn_binary.pt')
(paths.processed / 'cnn_metrics.json').write_text(json.dumps(metrics.__dict__, indent=2))
print('saved CNN weights + metrics')